# UncertainScene infrastructure smoke test (Colab or Kaggle)

**Looking for the scene01 experiments?** They are on branch `exp/change/scene01-heatmaps`: [open scene01_change_heatmaps.ipynb in Colab](https://colab.research.google.com/github/FinPThomas/UncertainScene/blob/exp/change/scene01-heatmaps/notebooks/scene01_change_heatmaps.ipynb).

This notebook only checks that a session can push results to GitHub, and is the template for new experiment notebooks.

The one notebook every session runs. See `docs/COLAB.md`.

1. Secrets: `GITHUB_TOKEN` (fine-grained, Contents read/write on UncertainScene) with notebook access on; optional `UNCERTAINSCENE_ACCOUNT` label.
2. Edit the config cell, then **Runtime → Run all**.
3. Session died? Open this notebook in any account and Run all again: finished steps are skipped.

In [ ]:
# ---- Config ---------------------------------------------------------------------------
BRANCH = "main"            # e.g. "exp/recon/mast3r-joint" (see docs/BRANCHING.md)
RUN_ID = "2026-10-09_smoke" # YYYY-MM-DD_<short-name>; reuse the same id to resume
PATH_NAME = "infra"         # research path: capture, recon, change, reid, deform, reach, report, infra
EXPECTED_COMMIT = ""        # optional: short sha you expect; setup stops if HEAD differs
RUN_SMOKE_TEST = True       # tiny step that checks guard + publish end to end
REPO = "FinPThomas/UncertainScene"

In [ ]:
# ---- Setup: token, clone to local disk, commit check, push check ------------------------
import base64, os, subprocess, sys
from pathlib import Path

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

token = secret("GITHUB_TOKEN")
assert token, "Add GITHUB_TOKEN in the Secrets panel and turn on notebook access"
os.environ["GITHUB_TOKEN"] = token
label = secret("UNCERTAINSCENE_ACCOUNT")
if label:
    os.environ["UNCERTAINSCENE_ACCOUNT"] = label

ROOT = Path("/kaggle/working" if os.path.isdir("/kaggle/working") else "/content")
REPO_DIR = ROOT / "UncertainScene"
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
AUTH = ["-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {auth}"]  # never stored in .git/config

def git(*args, cwd=REPO_DIR):
    p = subprocess.run(["git", *AUTH, *args], cwd=cwd, capture_output=True, text=True)
    if p.returncode:
        raise RuntimeError(p.stderr.replace(token, "***").replace(auth, "***"))
    return p.stdout.strip()

if not (REPO_DIR / ".git").exists():
    git("clone", "--branch", BRANCH, f"https://github.com/{REPO}.git", str(REPO_DIR), cwd=ROOT)
else:
    git("fetch", "origin", BRANCH)
    git("checkout", "-B", BRANCH, f"origin/{BRANCH}")
head = git("rev-parse", "--short", "HEAD")
print(f"On {BRANCH} at {head}:", git("log", "-1", "--format=%s"))
if EXPECTED_COMMIT and not head.startswith(EXPECTED_COMMIT[:7]):
    raise SystemExit(f"Stale code: HEAD is {head}, expected {EXPECTED_COMMIT}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR / "src"))

from uncertainscene.infra.publish import Publisher
pub = Publisher(REPO_DIR, BRANCH, token)
pub.ensure_identity()
pub.check_push_access()
print("Push access OK")

In [ ]:
# ---- Start the run: guard + resumable state ---------------------------------------------
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)

from uncertainscene.infra import Run
from uncertainscene.infra.guard import read_usage

run = Run.from_env(run_id=RUN_ID, path=PATH_NAME, branch=BRANCH)
print("Session label:", run.account)
print("Machine:", read_usage(run.budget.work_dir).line())
print("Run state:\n" + run.state.summary())

In [ ]:
# ---- Steps ------------------------------------------------------------------------------
# Each step is fn(out, work) -> optional one-line note.
#   out:  results/<path>/<run_id>/<step>/ in the repo (small files, committed for you)
#   work: scratch folder on local disk (big files; deleted when the step ends)
# Declare what the step needs; it will not start unless it can finish this session.
#
#   run.step("reconstruct_pair01", reconstruct, needs=dict(disk_gb=8, ram_gb=6, minutes=40))
#
# Big outputs: from uncertainscene.infra.store import Store, repo_slug
#   entry = Store(repo_slug(REPO_DIR)).put(work / "cloud.ply", tag=f"run-{RUN_ID}")

import json, time

def smoke(out, work):
    (work / "scratch.bin").write_bytes(b"0" * 1024)          # scratch is cleaned up afterwards
    run.guard.check()                                         # what long loops should call
    (out / "usage.json").write_text(json.dumps({"usage": read_usage(run.budget.work_dir).line(),
                                                "time": time.ctime()}, indent=2))
    return "guard and publish OK"

if RUN_SMOKE_TEST:
    print(run.step("smoke", smoke, needs=dict(disk_gb=0.1, ram_gb=0.1, minutes=1)))

In [ ]:
# ---- Finish -----------------------------------------------------------------------------
run.close()
print(run.state.summary())